# MossFormer

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from clear_memory import clear_memory

# 过滤警告
warnings.filterwarnings('ignore')

PyTorch版本: 2.8.0+cu128
CUDA可用: True
MPS可用: False


In [2]:
# 输入和输出目录
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-MossFormer')

# 获取文件列表
control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

print(f"Control组文件数: {len(control_files)}")
print(f"Dementia组文件数: {len(dementia_files)}")

Control组文件数: 242
Dementia组文件数: 309


## Load MossFormer Model

In [ ]:
from clearvoice import ClearVoice

# 使用 MossFormer 模型
model_name = 'MossFormerGAN_SE_16K'
target_sr = 16000  # 目标采样率

myClearVoice = ClearVoice(
    task='speech_enhancement',
    model_names=[model_name]
)

加载模型: MossFormerGAN_SE_16K...
注意: 首次使用会自动下载模型，可能需要几分钟...
✓ MossFormer 模型加载完成


## Denoise Function

In [5]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    使用 MossFormer 进行语音降噪和增强
    
    Args:
        audio_path: 输入音频文件路径
        model: ClearVoice 模型实例
        target_sr: 目标采样率（16000）
    
    Returns:
        denoised_audio: 降噪后的音频 numpy array
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # 如果是多声道，先转为单声道（在重采样之前）
    if len(audio.shape) == 2:
        audio = np.mean(audio, axis=1)
    
    # 重采样到目标采样率（使用 scipy，更稳定）
    if sr != target_sr:
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)
    
    # 确保是 float32 类型
    audio = audio.astype(np.float32)
    
    # 转换为 [batch, length] 格式
    audio = np.reshape(audio, [1, audio.shape[0]])
    
    # 应用 MossFormer 降噪
    with torch.no_grad():
        output_wav = model(audio, online_write=False)
    
    # output_wav 形状: [batch, length]
    return output_wav[0, :], target_sr

In [ ]:
def batch_denoise(files, output_subdir, model, target_sr, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: ClearVoice 模型实例
        target_sr: 目标采样率
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"降噪处理 {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            clear_memory()
            
            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, model, target_sr)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1
            
            del denoised_audio
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\n✗ 处理失败: {audio_file.name}: {e}")
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Denoise

In [7]:
batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files,
    output_dir / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)

初始显存状态: CUDA - 已分配: 0.02 GB, 已保留: 0.03 GB
清理后显存: CUDA - 已分配: 0.02 GB, 已保留: 0.03 GB


降噪处理 Dementia: 100%|██████████| 309/309 [38:35<00:00,  7.49s/it] 



Dementia 处理完成:
成功: 260
跳过: 49
失败: 0
总计: 309


降噪处理 Control: 100%|██████████| 242/242 [17:14<00:00,  4.27s/it]


Control 处理完成:
成功: 242
跳过: 0
失败: 0
总计: 242

✓ 所有处理完成！
总耗时: 55.82 分钟 (3349.46 秒)
输出目录: ../ad_detection/data/denoised/Pitt-MossFormer
最终显存状态: CUDA - 已分配: 0.03 GB, 已保留: 0.04 GB
